# LLM Experimentation Notebook
### Tokenization, Embeddings, Semantic Search & API Parameter Experiments

**Instructions:**
- Sections 1–2 (Tokenization, Embeddings) run fully offline using open-source libraries — no API key needed.
- Section 3 (LLM API experimentation) requires an API key (Groq). Instructions are provided; skip/mock if you don't have one, and use a Playground UI instead (see exercises sheet).
- Fill in the `# TODO` cells yourself, run each cell, and write a short observation in the markdown cell provided after each exercise.


## Import Required Libraries

In [4]:
import configparser
import os
import tiktoken
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from huggingface_hub import InferenceClient
from dotenv import load_dotenv
from jinja2 import Template

load_dotenv()

False

## Section 1 — Tokenization

**Goal:** See how text gets broken into tokens, and how token count varies by content and model.


In [5]:
# GPT-style tokenizer (cl100k_base is used by GPT-3.5/4 family; good general-purpose example)
enc = tiktoken.get_encoding("cl100k_base")

sample_sentences = [
    "Generative AI is transforming how we build software.",
    "supercalifragilisticexpialidocious",
    "The transformer architecture uses self-attention.",
    "नमस्ते, आप कैसे हैं?",       # non-English example
    "def add(a, b):\n    return a + b"  # code example
]

for s in sample_sentences:
    tokens = enc.encode(s)
    print(f"Text: {s!r}")
    print(f"  Token count: {len(tokens)}")
    print(f"  Tokens (decoded individually): {[enc.decode([t]) for t in tokens]}")
    print()


Text: 'Generative AI is transforming how we build software.'
  Token count: 10
  Tokens (decoded individually): ['Gener', 'ative', ' AI', ' is', ' transforming', ' how', ' we', ' build', ' software', '.']

Text: 'supercalifragilisticexpialidocious'
  Token count: 11
  Tokens (decoded individually): ['sup', 'erc', 'al', 'if', 'rag', 'il', 'istic', 'exp', 'ial', 'id', 'ocious']

Text: 'The transformer architecture uses self-attention.'
  Token count: 8
  Tokens (decoded individually): ['The', ' transformer', ' architecture', ' uses', ' self', '-', 'attention', '.']

Text: 'नमस्ते, आप कैसे हैं?'
  Token count: 20
  Tokens (decoded individually): ['न', 'म', 'स', '्�', '�', 'े', ',', ' �', '�', 'प', ' क', '�', '�', 'स', 'े', ' ह', '�', '�', 'ं', '?']

Text: 'def add(a, b):\n    return a + b'
  Token count: 11
  Tokens (decoded individually): ['def', ' add', '(a', ',', ' b', '):\n', '   ', ' return', ' a', ' +', ' b']



**Exercise 1.1:** Add 3 of your own sentences to `sample_sentences` above (try one long technical sentence, one sentence with an uncommon/made-up word, and one in a language other than English). Rerun the cell.

**Reflection (write here):** Which sentence had the most tokens relative to its word count? Why do you think that happened? _(TODO: your answer)_


In [6]:
# TODO: Exercise 1.2 — Compare token count vs. word count
# For each sentence in sample_sentences, print: word_count, token_count, and the ratio (tokens/word)

for s in sample_sentences:
    tokens = enc.encode(s)
    word_count = len(s.split())
    token_count = len(tokens)
    ratio = token_count / max(word_count, 1)
    print(f"{s[:40]!r:45} words={word_count:3}  tokens={token_count:3}  ratio={ratio:.2f}")


'Generative AI is transforming how we bui'    words=  8  tokens= 10  ratio=1.25
'supercalifragilisticexpialidocious'          words=  1  tokens= 11  ratio=11.00
'The transformer architecture uses self-a'    words=  5  tokens=  8  ratio=1.60
'नमस्ते, आप कैसे हैं?'                        words=  4  tokens= 20  ratio=5.00
'def add(a, b):\n    return a + b'            words=  7  tokens= 11  ratio=1.57


## Section 2 — Embeddings & Semantic Search

**Goal:** Generate embeddings for sentences and use cosine similarity to find semantically related sentences — the intuition behind semantic search (Week 2 preview).


In [7]:
# A small, fast open-source embedding model — good for classroom use
model = SentenceTransformer('all-MiniLM-L6-v2')

sentences = [
    "I love hiking in the mountains.",
    "The stock market fell sharply today.",
    "My dog loves to play fetch in the park.",
    "Interest rates rose this quarter.",
    "We went camping last weekend near the lake."
]

embeddings = model.encode(sentences)
print("Embedding shape per sentence:", embeddings.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape per sentence: (5, 384)


In [8]:
# embeddings[0]

In [9]:
# Compute pairwise cosine similarity matrix
from sklearn.metrics.pairwise import cosine_similarity
sim_matrix = cosine_similarity(embeddings)

print("Cosine Similarity Matrix:\n")
print("      " + "  ".join([f"S{i+1}" for i in range(len(sentences))]))
for i, row in enumerate(sim_matrix):
    print(f"S{i+1}  " + "  ".join([f"{v:.2f}" for v in row]))

print()
for i, s in enumerate(sentences):
    print(f"S{i+1}: {s}")


Cosine Similarity Matrix:

      S1  S2  S3  S4  S5
S1  1.00  -0.07  0.23  0.08  0.36
S2  -0.07  1.00  -0.01  0.28  -0.04
S3  0.23  -0.01  1.00  0.04  0.25
S4  0.08  0.28  0.04  1.00  0.05
S5  0.36  -0.04  0.25  0.05  1.00

S1: I love hiking in the mountains.
S2: The stock market fell sharply today.
S3: My dog loves to play fetch in the park.
S4: Interest rates rose this quarter.
S5: We went camping last weekend near the lake.


**Exercise 2.1 (Part H from exercises sheet):** Look at the similarity matrix above.
- Which pair of sentences has the highest similarity (excluding a sentence with itself)?
- Does this match your intuition? _(TODO: your answer)_

**Exercise 2.2:** Add 2 new sentences of your own — one that should be semantically close to an existing sentence, and one that should be far from all of them. Rerun Section 2 and confirm your prediction.


In [10]:
# TODO: Exercise 2.2 — add your own sentences and rerun
my_sentences = sentences + [
    # "TODO: your sentence 1",
    # "TODO: your sentence 2",
]

my_embeddings = model.encode(my_sentences)
my_sim_matrix = cosine_similarity(my_embeddings)

for i, row in enumerate(my_sim_matrix):
    print(f"S{i+1}  " + "  ".join([f"{v:.2f}" for v in row]))


S1  1.00  -0.07  0.23  0.08  0.36
S2  -0.07  1.00  -0.01  0.28  -0.04
S3  0.23  -0.01  1.00  0.04  0.25
S4  0.08  0.28  0.04  1.00  0.05
S5  0.36  -0.04  0.25  0.05  1.00


In [11]:
# TODO: Exercise 2.3 — Simple semantic search function
# Given a query sentence, find the most similar sentence from `sentences` using cosine similarity.

def semantic_search(query, corpus, corpus_embeddings, top_k=1):
    query_embedding = model.encode([query])
    sims = cosine_similarity(query_embedding, corpus_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(corpus[i], sims[i]) for i in top_idx]

# Try it out
query = "outdoor adventure in nature"  # TODO: try your own queries too
results = semantic_search(query, sentences, embeddings, top_k=3)
for text, score in results:
    print(f"{score:.3f}  |  {text}")


0.565  |  I love hiking in the mountains.
0.451  |  We went camping last weekend near the lake.
0.326  |  My dog loves to play fetch in the park.


**Reflection:** Try a query using a synonym or related concept that does NOT share exact keywords with any sentence (e.g., "financial markets" instead of "stock market"). Does semantic search still find the right sentence? Compare this to what a simple keyword search (`if word in sentence`) would have found. _(TODO: your answer)_


## Section 3 — LLM API Experimentation (Parameters)

**Goal:** Directly observe how `temperature`, `max_tokens`, and system prompts affect model output.

> **Note:** This section requires an API key. Set it as an environment variable before running (`GROQ_API_KEY`)and log your observations.


In [12]:
# Make sure _API_KEY is set in your environment before running this cell
hf_token = os.getenv("HF_TOKEN")

In [20]:
import os
import configparser
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

# Load environment variables
load_dotenv()

# Get Hugging Face token
hf_token = os.getenv("HF_TOKEN")

# Create a ConfigParser object
config = configparser.ConfigParser()

# Create the configuration directly
config["LLM_MODEL"] = {
    "model": "Qwen/Qwen2.5-7B-Instruct",
    "hub_timeout": "120"
}

# Create the InferenceClient
llm = InferenceClient(
    model=config.get("LLM_MODEL", "model"),
    token=hf_token,
    timeout=config.getint("LLM_MODEL", "hub_timeout")
)

print("LLM client created successfully!")

LLM client created successfully!


In [21]:
llm = InferenceClient(
    model=config.get("LLM_MODEL", "model"),
    token=hf_token,
    timeout=config.getint("LLM_MODEL", "hub_timeout")
)

In [22]:
def ask_hf(prompt, model, temperature=1.0, max_tokens=200):
    output = llm.chat_completion(
        messages=prompt,
        max_tokens=max_tokens,
        temperature=temperature
    )
    return output.choices[0].message.content

### Exercise 3.1 — Temperature

Run the same creative prompt at three different temperatures. Compare creativity vs. consistency (run each setting 2–3 times).


In [23]:
prompt = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Explain quantum computing in one sentence."}
        ]

for temp in [0.0, 0.7, 1.2]:
    print(f"--- Temperature = {temp} ---")
    for i in range(2):
        # TODO: uncomment once your API key is set
        # output = ask_hf(prompt=prompt, model=llm, temperature=temp, max_tokens=60)
        # print(output)
        pass


--- Temperature = 0.0 ---
--- Temperature = 0.7 ---
--- Temperature = 1.2 ---


**Reflection:** At temperature 0, did repeated runs give (near-)identical outputs? What changed as temperature increased? _(TODO: your answer)_


### Exercise 3.2 — Max Tokens

Ask for a detailed explanation with a very low `max_tokens` limit, then a higher one. Observe where the output gets cut off.


In [24]:
prompt = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Explain how transformers use self-attention, in detail."}
        ]

# TODO: uncomment once your API key is set
# short_output = ask_hf(prompt=prompt, model=llm, max_tokens=20)
# long_output = ask_hf(prompt=prompt, model=llm, max_tokens=300)
# print("SHORT (max_tokens=20):\n", short_output)
# print("\nLONG (max_tokens=300):\n", long_output)


### Exercise 3.3 — System Prompt vs. User Prompt

Set a system-level instruction and send several different user questions to see if it's consistently followed.


In [25]:
system_instruction = "Always answer in exactly one sentence, no matter the question."

questions = [
    "What is a transformer model?",
    "Why do we need tokenization?",
    "What is the difference between pretraining and fine-tuning?",
]

for q in questions:
    # TODO: uncomment once your API key is set
    prompt = [
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": q}
        ]
    # output = ask_hf(prompt=prompt, model=llm, max_tokens=100)
    # print(f"Q: {q}\nA: {output}\n")
    pass


**Reflection:** Was the system instruction followed consistently across all three questions? Note any cases where it was ignored or only partially followed. _(TODO: your answer)_

---

## Wrap-Up

Write a 3–5 sentence summary of what you learned about tokenization, embeddings, and LLM parameters, and one open question you still have.

_(TODO: your summary)_


Tokenization breaks text into small pieces called tokens so the LLM can understand it. Embeddings turn words or sentences into numbers that represent their meaning. LLM parameters like temperature control how the model responds.